# 🔬 viMSam MicroSAM Pipeline

This notebook runs the `vimsam_segmenter` package from segmenter_app.

The supported interfaces are:
- `vimsam-segmenter` (CLI command)
- `python /content/segmenter_app/main.py` (local entry point)
- Python API through `vimsam_segmenter` via `SegmenterApp().run(config)`

In [ ]:
# ============================================================
# 1. Colab Environment Setup
# ============================================================

# Install condacolab to enable conda/mamba support in Colab.
!pip install -q condacolab

import condacolab

# This installs a conda environment into the Colab runtime.
# Colab may restart the runtime after this cell.
condacolab.install()

In [ ]:
# ============================================================
# 2. Verify Conda Environment
# ============================================================

import condacolab

condacolab.check()

In [ ]:
# ============================================================
# 3. Dependency Installation
# ============================================================
#
# The original script pins NumPy below 2.0 for compatibility.
# PyTorch is pinned to 2.1.0 as in the original Colab script.
#
# Installed packages:
#   - micro_sam
#   - pytorch
#   - torchvision
#   - imageio
#   - imageio-ffmpeg
#   - tifffile
#   - python-dotenv
#   - pandas
#   - scikit-image
#   - ffmpeg
# ============================================================

# Remove any existing conda pinning that could conflict with dependency resolution.
!rm -f /usr/local/conda-meta/pinned

# Install NumPy version used by the original environment.
!mamba install -c conda-forge "numpy=1.26.4" -y

# Install MicroSAM and required Python dependencies.
!mamba install -c conda-forge \
    micro_sam \
    pytorch=2.1.0 \
    torchvision \
    imageio \
    imageio-ffmpeg \
    tifffile \
    python-dotenv \
    pandas \
    scikit-image \
    -y

# Install system ffmpeg.
!apt-get update && apt-get install -y ffmpeg


In [ ]:
# ============================================================
# 4. Runtime Verification
# ============================================================
#
# Verifies:
#   - PyTorch version
#   - CUDA/GPU availability
#   - NumPy version
#   - ffmpeg availability
# ============================================================

import os
import shutil
import subprocess

import torch
import numpy as np

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU is not available. The notebook will run on CPU, but segmentation may be slow.")

print(f"✅ NumPy version: {np.__version__}")

ffmpeg_path = shutil.which("ffmpeg")
print(f"✅ ffmpeg path: {ffmpeg_path if ffmpeg_path else 'Not found'}")

# Ensure imageio uses the system ffmpeg binary, matching the behavior in main.py.
if ffmpeg_path:
    os.environ["IMAGEIO_FFMPEG_EXE"] = ffmpeg_path
    print(f"✅ IMAGEIO_FFMPEG_EXE set to: {os.environ['IMAGEIO_FFMPEG_EXE']}")

In [ ]:
# ============================================================
# 5. Repository Setup and Editable Install
# ============================================================
#
# Clone the repository, move segmenter_app to /content,
# and install it in editable mode so that the vimsam-segmenter
# console command is available.
# ============================================================

import os
import shutil

REPOSITORY_URL = "https://github.com/Ilmars-Viksne/viMSam.git"
REPOSITORY_DIR = "/content/viMSam"
PROJECT_DIR = "/content/segmenter_app"

# Clean up any previous copies to make the notebook rerunnable.
if os.path.exists(REPOSITORY_DIR):
    shutil.rmtree(REPOSITORY_DIR)

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Clone the repository quietly.
!git clone -q https://github.com/Ilmars-Viksne/viMSam.git /content/viMSam

# Move only the segmentation application to /content.
!mv /content/viMSam/segmenter_app /content/segmenter_app

# Remove the rest of the repository.
!rm -rf /content/viMSam

# Install the app package in editable mode.
# This makes the vimsam-segmenter console command available.
%cd /content/segmenter_app
!pip install -e .

print(f"✅ Project prepared and installed at: {PROJECT_DIR}")

In [ ]:
# ============================================================
# 6. Verify Package and CLI
# ============================================================
#
# Verify that the package was installed correctly and
# the CLI commands are available.
# ============================================================

!vimsam-segmenter --help

print("\n" + "="*60 + "\n")

!python /content/segmenter_app/main.py --help

In [ ]:
# ============================================================
# 8. Inspect Project Structure
# ============================================================

!find /content/segmenter_app -maxdepth 3 -type f | sort | head -100

In [ ]:
# ============================================================
# 9. Execution Configuration
# ============================================================
#
# Define paths and parameters for the workflows.
# Modify these values if you want to run different inputs.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path("/content/segmenter_app")

MAIN_SCRIPT = PROJECT_DIR / "main.py"

RAW_SINGLE_INPUT = PROJECT_DIR / "data/input/images/20260226213602.raw"
RAW_SINGLE_OUTPUT = PROJECT_DIR / "data/output/images/res.png"

RAW_TIMESERIES_INPUT = PROJECT_DIR / "data/input/videos"
RAW_TIMESERIES_OUTPUT = PROJECT_DIR / "data/output/videos"

PROMPT_POINTS = "500,480"
TRACKING_METHOD = "pole"

print("Configuration")
print("-------------")
print(f"Main script:           {MAIN_SCRIPT}")
print(f"Raw single input:      {RAW_SINGLE_INPUT}")
print(f"Raw single output:     {RAW_SINGLE_OUTPUT}")
print(f"Raw timeseries input:  {RAW_TIMESERIES_INPUT}")
print(f"Raw timeseries output: {RAW_TIMESERIES_OUTPUT}")
print(f"Prompt points:         {PROMPT_POINTS}")
print(f"Tracking method:       {TRACKING_METHOD}")

In [ ]:
# ============================================================
# 10. Input Validation
# ============================================================
#
# This only prints useful diagnostics before running the CLI.
# ============================================================

from pathlib import Path

def report_path_status(path: Path, expected_type: str) -> None:
    """
    Print a simple status message for a file or directory path.
    """
    if expected_type == "file":
        exists = path.is_file()
    elif expected_type == "directory":
        exists = path.is_dir()
    else:
        exists = path.exists()

    status = "✅ Found" if exists else "⚠️ Not found"
    print(f"{status}: {path}")

report_path_status(MAIN_SCRIPT, "file")
report_path_status(RAW_SINGLE_INPUT, "file")
report_path_status(RAW_TIMESERIES_INPUT, "directory")

In [ ]:
# ============================================================
# 11. Run Workflow: Raw Single Image
# ============================================================
#
# Use the installed vimsam-segmenter CLI command.
# ============================================================

!vimsam-segmenter \
    --input "/content/segmenter_app/data/input/images/20260226213602.raw" \
    --out "/content/segmenter_app/data/output/images/res.png" \
    --workflow "raw_single" \
    --points "500,480" \
    --show-prompts \
    --save-combined

In [ ]:
# ============================================================
# 12. Visualize Raw Single Image Output
# ============================================================

from IPython.display import Image, display
import os

if os.path.exists(RAW_SINGLE_OUTPUT):
    display(Image(filename=str(RAW_SINGLE_OUTPUT)))
else:
    print(f"⚠️ Output image not found: {RAW_SINGLE_OUTPUT}")

In [ ]:
# ============================================================
# 13. Run Workflow: Raw Time Series
# ============================================================
#
# Use the installed vimsam-segmenter CLI command.
# ============================================================

!vimsam-segmenter \
    --input "/content/segmenter_app/data/input/videos/" \
    --out "/content/segmenter_app/data/output/videos/" \
    --workflow "raw_timeseries" \
    --points "500,480" \
    --tracking-method "pole" \
    --show-prompts \
    --save-combined

In [ ]:
# ============================================================
# 14. Inspect Generated Outputs
# ============================================================

print("Image outputs:")
!find /content/segmenter_app/data/output/images -maxdepth 2 -type f | sort

print("\nVideo / time series outputs:")
!find /content/segmenter_app/data/output/videos -maxdepth 2 -type f | sort

In [ ]:
# ============================================================
# 16. Optional: Package Outputs
# ============================================================
#
# These commands are optional and preserve the behavior of the
# commented-out commands in the original script.
# ============================================================

# Zip only generated data.
!zip -r data.zip /content/segmenter_app/data

# Zip the full segmentation application.
#!zip -r segmenter_app.zip /content/segmenter_app

In [ ]:
'''
!vimsam-segmenter \
  --input data/input/images/20260226213602.raw \
  --out data/output/images/res.png \
  --workflow raw_single \
  --box "450,430,560,540" \
  --show-prompts \
  --save-combined
  '''

In [ ]:
# Test in Terminal
'''
pip install pytest
python -m pytest segmenter_app/tests/test_config_and_cli.py -q
python -m pytest segmenter_app/tests/test_geometry.py -q
python -m pytest segmenter_app/tests/test_local_io.py -q
python -m pytest segmenter_app/tests/test_model_service.py -q
python -m pytest segmenter_app/tests/test_raw_io.py -q
python -m pytest segmenter_app/tests/test_visualization.py -q
'''